In [14]:
import os
import time
import json
import re
import concurrent.futures
import requests
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    ScoringProfile,
    TextWeights,
    FreshnessScoringParameters,
    MagnitudeScoringParameters,
    TagScoringParameters,
    ScoringFunction,
    FreshnessScoringFunction,
    MagnitudeScoringFunction,
    TagScoringFunction
)

# Import Azure OpenAI for tokenization estimation
try:
    from tiktoken import encoding_for_model, get_encoding
    TIKTOKEN_AVAILABLE = True
except ImportError:
    print("Warning: tiktoken package not installed. Token estimation will not be available.")
    TIKTOKEN_AVAILABLE = False

# --- Configuration ---
# Load environment variables from .env file
load_dotenv()

# Retrieve configuration from environment variables
# Ensure these are set in your environment or .env file
service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
api_key = os.getenv("AZURE_SEARCH_API_KEY")

# --- Token Tracking Configuration ---
ENABLE_TOKEN_TRACKING = True  # Set to False to disable token tracking
TOKEN_ENCODER = "cl100k_base"  # Encoding to use for token estimation
TOKEN_TRACKING_METHOD = "estimate"  # 'estimate' or 'api' (if you have API access to exact counts)

# --- Token Tracking Functions ---
class TokenTracker:
    """Class to track token usage across different search methods."""
    
    def __init__(self, enable=ENABLE_TOKEN_TRACKING, encoder_name=TOKEN_ENCODER):
        self.enable = enable
        self.encoder_name = encoder_name
        self.search_token_usage = {}
        
        if enable and TIKTOKEN_AVAILABLE:
            try:
                self.encoder = get_encoding(encoder_name)
                print(f"Token tracking initialized with encoder: {encoder_name}")
            except Exception as e:
                print(f"Warning: Could not initialize token encoder: {e}")
                self.enable = False
        else:
            self.enable = enable and TIKTOKEN_AVAILABLE
    
    def _estimate_tokens(self, text):
        """Estimate the number of tokens in a string using the encoder."""
        if not self.enable or not text:
            return 0
        
        try:
            return len(self.encoder.encode(text))
        except Exception as e:
            print(f"Error estimating tokens: {e}")
            return 0
    
    def track_search_request(self, method_name, query_text, options=None, response_data=None):
        """
        Track token usage for a search request.
        
        Args:
            method_name (str): Name of the search method
            query_text (str): The search query text
            options (dict, optional): Search options and parameters
            response_data (list, optional): Search response data
        """
        if not self.enable:
            return
        
        # Create entry for this method if it doesn't exist
        if method_name not in self.search_token_usage:
            self.search_token_usage[method_name] = {
                'requests': 0,
                'query_tokens': 0,
                'options_tokens': 0,
                'response_tokens': 0,
                'total_tokens': 0
            }
        
        # Track request count
        self.search_token_usage[method_name]['requests'] += 1
        
        # Estimate query tokens
        query_tokens = self._estimate_tokens(query_text)
        self.search_token_usage[method_name]['query_tokens'] += query_tokens
        
        # Estimate options tokens
        options_tokens = 0
        if options:
            options_tokens = self._estimate_tokens(json.dumps(options))
            self.search_token_usage[method_name]['options_tokens'] += options_tokens
        
        # Estimate response tokens
        response_tokens = 0
        if response_data:
            # For search results, we need to serialize to estimate tokens
            response_sample = response_data[:5] if len(response_data) > 5 else response_data
            response_str = json.dumps(response_sample)
            # Estimate based on the sample and scale to full response size
            tokens_per_result = self._estimate_tokens(response_str) / max(len(response_sample), 1)
            response_tokens = int(tokens_per_result * len(response_data))
            self.search_token_usage[method_name]['response_tokens'] += response_tokens
        
        # Track total tokens
        total_tokens = query_tokens + options_tokens + response_tokens
        self.search_token_usage[method_name]['total_tokens'] += total_tokens
        
        # Print debug info for this request
        if self.enable:
            print(f"\nToken usage for {method_name}:")
            print(f"  Query tokens: {query_tokens}")
            print(f"  Options tokens: {options_tokens}")
            print(f"  Response tokens: {response_tokens}")
            print(f"  Total tokens for this request: {total_tokens}")
    
    def get_token_summary(self):
        """Get a summary of token usage across all tracked methods."""
        if not self.enable or not self.search_token_usage:
            return None
        
        summary = {
            'methods': self.search_token_usage,
            'total_tokens': sum(method['total_tokens'] for method in self.search_token_usage.values()),
            'total_requests': sum(method['requests'] for method in self.search_token_usage.values())
        }
        
        return summary
    
    def print_token_summary(self):
        """Print a human-readable summary of token usage."""
        if not self.enable or not self.search_token_usage:
            print("Token tracking not enabled or no data available.")
            return
        
        print("\n===== Token Usage Summary =====")
        print(f"{'Method':<25} {'Requests':<10} {'Query':<10} {'Options':<10} {'Response':<12} {'Total'}")
        print("-" * 75)
        
        total_tokens = 0
        for method_name, usage in self.search_token_usage.items():
            method_total = usage['total_tokens']
            total_tokens += method_total
            print(f"{method_name:<25} {usage['requests']:<10} {usage['query_tokens']:<10} "
                  f"{usage['options_tokens']:<10} {usage['response_tokens']:<12} {method_total}")
        
        print("-" * 75)
        print(f"{'TOTAL':<25} {'':<10} {'':<10} {'':<10} {'':<12} {total_tokens}")
        print("\nNote: Token counts are estimates and may vary from actual usage.")

# Initialize token tracker
token_tracker = TokenTracker(enable=ENABLE_TOKEN_TRACKING)

# --- Update run_search_query to track tokens ---
def run_search_query(query_text, order_by=None, top_n=5, scoring_profile=None, scoring_parameters=None):
    """
    Runs a search query against the Azure AI Search index with optional scoring profile.
    Now includes token tracking.

    Args:
        query_text (str): The search term or phrase.
        order_by (str, optional): The field(s) to order results by.
        top_n (int): The maximum number of results to return.
        scoring_profile (str, optional): Name of the scoring profile to use.
        scoring_parameters (list, optional): List of scoring parameters for the profile.

    Returns:
        tuple: A tuple containing (list_of_results, duration_ms)
               Returns (None, 0) on error.
    """
    print("-" * 40)
    query_details = [f"'{query_text}'"]
    if scoring_profile:
        query_details.append(f"Scoring Profile: '{scoring_profile}'")
    if order_by:
        query_details.append(f"ORDER BY: '{order_by}'")
    query_details.append(f"Top: {top_n}")
    print(f"Running query: {' | '.join(query_details)}")
    print("-" * 40)

    # Prepare the search options for token tracking
    search_options = {
        "select": ["id", "content", "sourcefile"],
        "order_by": order_by,
        "top": top_n,
        "include_total_count": True,
        "scoring_profile": scoring_profile,
        "scoring_parameters": scoring_parameters
    }
    
    method_name = "Default Search"
    if scoring_profile:
        method_name = f"Search with {scoring_profile}"

    start_time = time.time()
    try:
        if order_by and '@search.score' in order_by:
            print("Note: Cannot directly sort by @search.score in the query.")
            print("Will retrieve results and sort them by score afterward.")
            order_by = None 
            search_options["order_by"] = None
        
        search_results = search_client.search(
            search_text=query_text,
            **{k: v for k, v in search_options.items() if v is not None}
        )

        results_list = []
        count = search_results.get_count()
        print(f"Total matching documents found: {count}")

        for result in search_results:
            # Adjust field names ('id', 'content', etc.) based on YOUR index schema
            doc_id = result.get('id', 'N/A') # Use .get for safety if field might be missing
            content_snippet = result.get('content', 'N/A')[:150] + "..." # Show snippet
            score = result['@search.score'] # Score is always available
            results_list.append(result) # Store the full result dict if needed later

            print(f"  Score: {score:.4f}")
            print(f"  ID:    {doc_id}")
            print(f"  Content: {content_snippet}\n")

        end_time = time.time()
        duration_ms = (end_time - start_time) * 1000
        print(f"Query Duration: {duration_ms:.2f} ms")
        
        # Track token usage for this search request
        if ENABLE_TOKEN_TRACKING:
            token_tracker.track_search_request(
                method_name=method_name,
                query_text=query_text,
                options=search_options,
                response_data=results_list
            )
            
        return results_list, duration_ms

    except Exception as e:
        print(f"An error occurred during search: {e}")
        return None, 0

# --- Update write_summary_to_txt to include token usage ---
def write_summary_to_txt(
    default_results, custom_scored_results, keyword_results, best_practices_results, adaptive_results,
    default_duration, bp_duration, adaptive_duration, search_query,
    include_semantic=False, semantic_results=None, semantic_duration=0, file_path="results_summary.txt",
    business_impact_metrics=None
):
    """
    Writes a concise summary of the scoring results and comparisons to a txt file,
    now including business impact metrics and token usage.
    """
    # ...existing code...
    summary_lines = []
    def get_top_result_info(results, method_name):
        if not results:
            return f"{method_name}: No results."
        top = max(results, key=lambda x: x.get('custom_score', x.get('@search.score', 0)))
        return (
            f"{method_name} Top Result:\n"
            f"  ID: {top.get('id', 'N/A')}\n"
            f"  Score: {top.get('custom_score', top.get('@search.score', 0)):.4f}\n"
            f"  Source: {top.get('sourcefile', 'N/A')}\n"
            f"  Snippet: {top.get('content', '')[:200]}...\n"
        )
    
    # Executive summary at the top
    summary_lines.append(f"EXECUTIVE SUMMARY - SEARCH RELEVANCE ANALYSIS")
    summary_lines.append(f"====================================================")
    summary_lines.append(f"Query: '{search_query}'")
    
    # Add business impact summary if available
    if business_impact_metrics:
        best_method = max(business_impact_metrics, key=lambda x: x['estimated_additional_revenue_yearly'])
        summary_lines.append(f"\nBUSINESS IMPACT HIGHLIGHTS:")
        summary_lines.append(f"✓ Recommended Approach: {best_method['method_name']}")
        summary_lines.append(f"✓ Estimated Annual Revenue Increase: ${best_method['estimated_additional_revenue_yearly']:,.2f}")
        summary_lines.append(f"✓ Relevance Improvement: {best_method['relevance_improvement_percentage']:.1f}% in top results\n")
    
    summary_lines.append(f"\nDETAILED RESULTS")
    summary_lines.append(f"====================================================")
    summary_lines.append(get_top_result_info(default_results, 'Default Scoring'))
    summary_lines.append(get_top_result_info(custom_scored_results, 'Custom Post-Query Scoring'))
    summary_lines.append(get_top_result_info(keyword_results, 'Keyword Boosting'))
    summary_lines.append(get_top_result_info(best_practices_results, 'Azure Best Practices'))
    summary_lines.append(get_top_result_info(adaptive_results, 'Adaptive Query Profile'))
    
    if include_semantic and semantic_results:
        summary_lines.append(get_top_result_info(semantic_results, 'Semantic Search'))
    
    # ...existing code (ranking comparison, performance summary, etc.)...
    
    # Add token usage summary if enabled
    if ENABLE_TOKEN_TRACKING:
        token_summary = token_tracker.get_token_summary()
        if token_summary:
            summary_lines.append("\nTOKEN USAGE SUMMARY:")
            summary_lines.append(f"{'Method':<25} {'Requests':<10} {'Query':<10} {'Options':<10} {'Response':<12} {'Total'}")
            summary_lines.append("-" * 75)
            
            for method_name, usage in token_summary['methods'].items():
                method_total = usage['total_tokens']
                summary_lines.append(f"{method_name:<25} {usage['requests']:<10} {usage['query_tokens']:<10} "
                      f"{usage['options_tokens']:<10} {usage['response_tokens']:<12} {method_total}")
            
            summary_lines.append("-" * 75)
            summary_lines.append(f"{'TOTAL':<25} {'':<10} {'':<10} {'':<10} {'':<12} {token_summary['total_tokens']}")
            summary_lines.append("\nNote: Token counts are estimates and may vary from actual usage.")
    
    # Write to file
    with open(file_path, "w", encoding="utf-8") as f:
        f.write("\n".join(summary_lines))
    print(f"\nComprehensive analysis written to {file_path}")

# --- Main Execution ---
if __name__ == "__main__":
    # Welcome message
    print("\n===== Azure AI Search Scoring Comparison Tool =====\n")
    print("This tool demonstrates different scoring mechanisms in Azure AI Search")
    print("and compares their impact on search result relevance and business outcomes.\n")
    
    # Ask for user's search query
    search_query = input("Enter your search query: ").strip()
    if not search_query:
        search_query = "What are my prescription benefits?"  # Default fallback
        print(f"Using default query: '{search_query}'")
    
    # Extract potential keywords from the query for later use
    keywords = [word for word in search_query.split() if len(word) > 3]
    
    # Optional business metrics configuration - customize for your business
    business_metrics = {
        'click_probability_top3': 0.78,  # 78% of users click on top 3 results
        'conversion_rate_relevant': 0.12,  # 12% conversion when finding relevant results
        'conversion_rate_irrelevant': 0.03,  # 3% conversion otherwise
        'avg_transaction_value': 85,  # Average transaction value ($)
        'search_volume_daily': 1000  # Daily search volume
    }
    
    print("\n===== Running Scoring Methods and Business Impact Analysis =====\n")
    
    # Store business impact metrics for later comparison
    impact_metrics = []
    
    # 1. Default Scoring
    print("\n\n===== Method 1: Default Azure AI Search Scoring =====")
    default_results, default_duration = run_search_query(search_query)
    print_results_with_custom_scores(default_results, "Default Scoring Results", detailed=True)
    
    # Evaluate default search relevance
    default_relevance = evaluate_search_relevance(default_results, query=search_query)
    print("\n=== Default Search Relevance Assessment ===")
    print(f"Source diversity: {default_relevance.get('source_diversity', 'N/A')} unique sources")
    print(f"Score differentiation: {default_relevance.get('differentiation', 'N/A')}")
    if 'estimated_relevance' in default_relevance:
        print(f"Estimated relevance: {default_relevance.get('estimated_relevance', 'N/A')}")
    
    # 2. Custom Post-Query Scoring
    print("\n\n===== Method 2: Custom Post-Query Scoring =====")
    custom_criteria = {
        'boost_factor': 1.2,
        'min_threshold': 3.0,
        'content_relevance_weight': 1.5
    }
    custom_scored_results = apply_custom_scoring(default_results.copy(), custom_criteria)
    print_results_with_custom_scores(custom_scored_results, "Custom Post-Query Scored Results", detailed=True)
    
    # Track token usage for custom post-query scoring
    if ENABLE_TOKEN_TRACKING:
        # Since this method applies scoring after querying, it doesn't make additional search requests
        # But we still want to track its token usage for client-side processing
        token_tracker.track_search_request(
            method_name="Custom Post-Query Scoring",
            query_text="", # No new query text, just reprocessing
            options=custom_criteria,
            response_data=custom_scored_results
        )
    
    # Calculate business impact for custom scoring
    custom_impact = calculate_business_impact(
        default_results, 
        custom_scored_results, 
        "Custom Post-Query Scoring",
        business_metrics
    )
    impact_metrics.append(custom_impact)
    
    # 3. Keyword Boosting (concise output)
    print("\n\n===== Method 3: Keyword Boosting =====")
    keyword_results = boost_keywords_in_results(default_results.copy(), keywords, 2.0)
    print_results_with_custom_scores(keyword_results, "Keyword Boosted Results", detailed=False)
    
    # Track token usage for keyword boosting
    if ENABLE_TOKEN_TRACKING:
        token_tracker.track_search_request(
            method_name="Keyword Boosting",
            query_text="", # No new query text
            options={"boost_factor": 2.0, "keywords": keywords},
            response_data=keyword_results
        )
    
    # Calculate business impact for keyword boosting
    keyword_impact = calculate_business_impact(
        default_results, 
        keyword_results, 
        "Keyword Boosting",
        business_metrics
    )
    impact_metrics.append(keyword_impact)
    
    # 4. Azure Search Best Practices (concise output)
    print("\n\n===== Method 4: Azure AI Search Best Practices Scoring Profile =====")
    bp_profile_created = apply_azure_search_best_practices()
    if bp_profile_created:
        best_practices_results, bp_duration = run_search_query(
            search_query, scoring_profile="azure_search_best_practices")
        print_results_with_custom_scores(best_practices_results, "Azure Best Practices Results", detailed=False)
        
        # Calculate business impact for best practices
        bp_impact = calculate_business_impact(
            default_results, 
            best_practices_results, 
            "Azure Best Practices",
            business_metrics
        )
        impact_metrics.append(bp_impact)
    else:
        best_practices_results, bp_duration = None, 0
        print("Skipping search with 'azure_search_best_practices' profile due to creation error.")
    
    # 5. Custom Scoring Profile with Query Keywords (concise output)
    print("\n\n===== Method 5: Custom Scoring Profile with Query Keywords =====")
    profile_name = "adaptive_query_profile"
    adaptive_profile_created = create_keyword_boosting_profile(
        profile_name=profile_name,
        field_weights={'content': 4.0},
        boost_keywords=keywords,
        keyword_field="content"
    )
    if adaptive_profile_created:
        adaptive_results, adaptive_duration = run_search_query(
            search_query, scoring_profile=profile_name)
        print_results_with_custom_scores(adaptive_results, "Adaptive Query Profile Results", detailed=False)
        
        # Calculate business impact for adaptive query profile
        adaptive_impact = calculate_business_impact(
            default_results, 
            adaptive_results, 
            "Adaptive Query Profile",
            business_metrics
        )
        impact_metrics.append(adaptive_impact)
    else:
        adaptive_results, adaptive_duration = None, 0
        print(f"Skipping search with '{profile_name}' profile due to creation error.")
    
    # --- Concise Ranking Comparison Summary ---
    print("\n\n===== Scoring Methods Ranking Comparison Summary =====\n")
    compare_result_rankings(default_results, custom_scored_results, "Custom Post-Query Scoring")
    compare_result_rankings(default_results, keyword_results, "Keyword Boosting")
    compare_result_rankings(default_results, best_practices_results, "Azure Best Practices Scoring")
    compare_result_rankings(default_results, adaptive_results, "Adaptive Query Profile")
    
    # --- Performance Summary ---
    print("\n===== Scoring Methods Performance Summary =====\n")
    print(f"{'Scoring Method':<30} {'Query Duration (ms)':<20} {'Top Result ID':<30} {'Est. Tokens'}")
    print("-" * 85)
    
    def get_top_result_id(results):
        if not results:
            return "N/A"
        if any('custom_score' in r for r in results):
            return max(results, key=lambda x: x.get('custom_score', 0)).get('id', 'N/A')
        return max(results, key=lambda x: x.get('@search.score', 0)).get('id', 'N/A')
    
    # Function to get token count for a method
    def get_token_count(method_name):
        if not ENABLE_TOKEN_TRACKING:
            return "N/A"
        summary = token_tracker.get_token_summary()
        if not summary or method_name not in summary['methods']:
            return "N/A"
        return summary['methods'][method_name]['total_tokens']
    
    print(f"{'1. Default Search':<30} {default_duration:<20.2f} {get_top_result_id(default_results):<30} {get_token_count('Default Search')}")
    print(f"{'2. Custom Post-Query':<30} {default_duration:<20.2f} {get_top_result_id(custom_scored_results):<30} {get_token_count('Custom Post-Query Scoring')}")
    print(f"{'3. Keyword Boosting':<30} {default_duration:<20.2f} {get_top_result_id(keyword_results):<30} {get_token_count('Keyword Boosting')}")
    print(f"{'4. Azure Best Practices':<30} {bp_duration:<20.2f} {get_top_result_id(best_practices_results):<30} {get_token_count('Search with azure_search_best_practices')}")
    print(f"{'5. Adaptive Query Profile':<30} {adaptive_duration:<20.2f} {get_top_result_id(adaptive_results):<30} {get_token_count('Search with adaptive_query_profile')}")
    
    # --- Business Impact Analysis ---
    print("\n===== Business Impact Analysis =====\n")
    print("Analyzing how search improvements translate to business value...")
    print("Using the following business metrics:")
    print(f"- Click probability on top 3 results: {business_metrics['click_probability_top3']:.0%}")
    print(f"- Conversion rate with relevant results: {business_metrics['conversion_rate_relevant']:.0%}")
    print(f"- Conversion rate with irrelevant results: {business_metrics['conversion_rate_irrelevant']:.0%}")
    print(f"- Average transaction value: ${business_metrics['avg_transaction_value']}")
    print(f"- Daily search volume: {business_metrics['search_volume_daily']:,}")
    
    # Print business impact comparison
    print_business_impact(impact_metrics, compare=True)
    
    # --- Token Usage Analysis ---
    if ENABLE_TOKEN_TRACKING:
        print("\n===== Token Usage Analysis =====\n")
        print("Analyzing token consumption across different scoring methods...")
        token_tracker.print_token_summary()
        
        # Identify method with lowest token usage that still performs well
        if token_tracker.search_token_usage:
            token_efficient_methods = sorted(
                [(name, usage['total_tokens']) for name, usage in token_tracker.search_token_usage.items()],
                key=lambda x: x[1]
            )
            
            if token_efficient_methods:
                most_efficient = token_efficient_methods[0]
                print(f"\nMost token-efficient method: {most_efficient[0]} with {most_efficient[1]} tokens")
                
                # Find the business impact of this method if available
                efficient_method_impact = next((im for im in impact_metrics if im['method_name'] in most_efficient[0]), None)
                if efficient_method_impact:
                    print(f"Business impact of this method:")
                    print(f"- Estimated yearly revenue increase: ${efficient_method_impact['estimated_additional_revenue_yearly']:,.2f}")
                    print(f"- Relevance improvement: {efficient_method_impact['relevance_improvement_percentage']:.1f}%")
    
    print("\n===== Analysis Complete =====\n")
    print("This analysis demonstrates how different scoring techniques")
    print("impact search relevance in Azure AI Search and quantifies")
    print("the potential business value and token efficiency of implementing these improvements.")
    
    # Write comprehensive summary to txt file with business impact metrics and token usage
    write_summary_to_txt(
        default_results, custom_scored_results, keyword_results, best_practices_results, adaptive_results,
        default_duration, bp_duration, adaptive_duration, search_query,
        business_impact_metrics=impact_metrics
    )

Token tracking initialized with encoder: cl100k_base

===== Azure AI Search Scoring Comparison Tool =====

This tool demonstrates different scoring mechanisms in Azure AI Search
and compares their impact on search result relevance and business outcomes.


===== Running Scoring Methods and Business Impact Analysis =====



===== Method 1: Default Azure AI Search Scoring =====
----------------------------------------
Running query: 'What are my prescription benefits' | Top: 5
----------------------------------------

===== Running Scoring Methods and Business Impact Analysis =====



===== Method 1: Default Azure AI Search Scoring =====
----------------------------------------
Running query: 'What are my prescription benefits' | Top: 5
----------------------------------------
Total matching documents found: 286
  Score: 7.8433
  ID:    northwind_standard_benefits_details_236
  Content: uments are the ultimate 
authority for any questions about benefits, coverage, and exclusions.  
The pl